# Grype SBOM and vulnerability output explorer

**Purpose:** Compare how the same vulnerability scan looks when Grype renders it in JSON, CycloneDX, and SARIF formats. This notebook runs a scan against a sample target, inspects each output structure, and highlights where each format shines or trips you up in a pipeline.

## Prerequisites

- `grype` installed (>=0.70). On macOS/Linux: `brew install anchore/grype/grype` or download the binary from [grype releases](https://github.com/anchore/grype/releases).
- A scan target: an image reference (e.g., `alpine:latest`), a directory, or a SBOM file.
- Python 3.8+ with `json`, `pathlib`, `subprocess`, and `os` from the standard library.

> CycloneDX output is produced here by feeding a Syft-generated SBOM into Grype so the SBOM format itself is visible before the vulnerability layer is added.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

WORK_DIR = Path("grype-output-demo")
WORK_DIR.mkdir(exist_ok=True)
print(f"Working directory: {WORK_DIR.resolve()}")

In [ ]:
"""
Detect the target. Use an env var if set, otherwise fall back to a tiny local image.
If nothing is available, create a trivial test fixture in the work dir.
"""
import os

TARGET = os.environ.get("GRYPE_TARGET", "alpine:latest")
print(f"Target: {TARGET}")

if TARGET.startswith("dir:") or (Path(TARGET).exists() and Path(TARGET).is_dir()):
    scan_target = f"dir:{TARGET}" if not TARGET.startswith("dir:") else TARGET
elif TARGET.endswith(".json"):
    scan_target = TARGET
else:
    scan_target = TARGET

print(f"Scan target string: {scan_target}")

In [ ]:
def run(cmd, label):
    """Run a command and return (returncode, stdout, stderr)."""
    print(f"\n>>> {label}")
    print(f"    Command: {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"    Return code: {r.returncode}")
        print(f"    Stderr: {r.stderr[:300]}")
    else:
        print(f"    Return code: {r.returncode}")
    return r.returncode, r.stdout, r.stderr

## Step 1: produce the native Grype JSON output

Grype's default structured output is JSON. It lists every match with the vulnerability metadata, the affected package, and the fix status. This format is the source of truth for anything downstream.

In [ ]:
json_path = WORK_DIR / "grype.json"
rc, stdout, stderr = run(
    ["grype", scan_target, "-o", "json", "-q"],
    "Grype JSON scan"
)
if rc == 0:
    json_path.write_text(stdout)
    print(f"Wrote {json_path.name} ({json_path.stat().st_size:,} bytes)")
else:
    print("JSON scan failed; creating a minimal fixture so the notebook still runs.")
    fixture = {
        "matches": [
            {
                "vulnerability": {"id": "CVE-2024-EXAMPLE", "severity": "High", "fixVersions": ["1.2.3"]},
                "matchDetails": [{"matchedPackage": {"name": "example-lib", "version": "1.2.0"}}]
            }
        ]
    }
    json_path.write_text(json.dumps(fixture, indent=2))
    print(f"Wrote fixture {json_path.name}")

## Step 2: produce the SARIF output

SARIF is the lingua franca for static analysis in GitHub Code Scanning and many enterprise tools. The same Grype scan re-rendered as SARIF reorganizes findings into runs → results → locations.

In [ ]:
sarif_path = WORK_DIR / "grype.sarif"
rc, stdout, stderr = run(
    ["grype", scan_target, "-o", "sarif", "-q"],
    "Grype SARIF scan"
)
if rc == 0:
    sarif_path.write_text(stdout)
    print(f"Wrote {sarif_path.name} ({sarif_path.stat().st_size:,} bytes)")
else:
    print("SARIF scan failed; creating a minimal fixture.")
    sarif_fixture = {
        "$schema": "https://json.schemastore.org/sarif-2.1.0.json",
        "version": "2.1.0",
        "runs": [
            {
                "tool": {"driver": {"name": "Grype", "version": "demo"}},
                "results": [
                    {
                        "ruleId": "CVE-2024-EXAMPLE",
                        "level": "warning",
                        "message": {"text": "example-lib 1.2.0 has a High severity vulnerability"},
                        "locations": [
                            {"physicalLocation": {"artifactLocation": {"uri": "unknown"}}}
                        ]
                    }
                ]
            }
        ]
    }
    sarif_path.write_text(json.dumps(sarif_fixture, indent=2))
    print(f"Wrote fixture {sarif_path.name}")

## Step 3: produce a CycloneDX SBOM as input

One way to see CycloneDX in a Grype workflow is to generate the SBOM first with Syft, then hand it to Grype. The resulting SBOM file has its own schema that pipelines consume independently of vulnerability data.

In [ ]:
cdx_path = WORK_DIR / "sbom.cdx.json"

if not shutil.which("syft"):
    print("syft not found; writing a minimal CycloneDX fixture.")
else:
    rc, stdout, stderr = run(
        ["syft", scan_target if not scan_target.startswith("dir:") else scan_target.replace("dir:", ""), "-o", "cyclonedx-json"],
        "Syft CycloneDX SBOM"
    )
    if rc == 0:
        cdx_path.write_text(stdout)
        print(f"Wrote {cdx_path.name} ({cdx_path.stat().st_size:,} bytes)")
    else:
        print("Syft run failed; writing a fixture.")

# ensure the fixture exists as a fallback
if not cdx_path.exists():
    cdx_fixture = {
        "bomFormat": "CycloneDX",
        "specVersion": "1.4",
        "components": [{
            "type": "application",
            "name": "example-lib",
            "version": "1.2.0"
        }]
    }
    cdx_path.write_text(json.dumps(cdx_fixture, indent=2))
    print(f"Wrote fixture {cdx_path.name}")

In [ ]:
import shutil

if not shutil.which("syft"):
    print("\nSyft is not installed; using the CycloneDX fixture we created earlier.")
else:
    print("\nSyft is available. The CycloneDX SBOM above was generated for the same target.")

## Step 4: compare the structures

Now load all three files and surface the key differences: where the vulnerability metadata lives, how severity is expressed, and how you would navigate each format in a script.

In [ ]:
grype_data = json.loads(json_path.read_text())
sarif_data = json.loads(sarif_path.read_text())
cdx_data = json.loads(cdx_path.read_text())

print("=== JSON top-level keys ===")
print(list(grype_data.keys())[:10])

print("\n=== SARIF run count ===")
print(len(sarif_data.get("runs", [])))

print("\n=== CycloneDX top-level keys ===")
print(list(cdx_data.keys())[:10])

In [ ]:
"""
Summarize matches across formats so we can see coverage numbers.
"""

def summarize_json(data):
    matches = data.get("matches", [])
    by_sev = {}
    for m in matches:
        sev = (
            m.get("vulnerability", {})
            .get("severity", "Unknown")
            .capitalize()
        )
        by_sev[sev] = by_sev.get(sev, 0) + 1
    return {"count": len(matches), "by_severity": by_sev}


def summarize_sarif(data):
    results = []
    for run in data.get("runs", []):
        results.extend(run.get("results", []))
    by_sev = {}
    for r in results:
        level = (r.get("level") or "warning").capitalize()
        by_sev[level] = by_sev.get(level, 0) + 1
    return {"count": len(results), "by_severity": by_sev}


print("=== Grype JSON summary ===")
print(json.dumps(summarize_json(grype_data), indent=2))

print("\n=== SARIF summary ===")
print(json.dumps(summarize_sarif(sarif_data), indent=2))

## What I found

1. **JSON** is the easiest to work with in scripts: every vulnerability is a flat-ish dict under `matches`, and each entry already exposes severity, fix versions, and the affected package together.
2. **SARIF** wraps findings in a `runs → results` hierarchy. It demands a bit more traversal (`sarif_data["runs"][0]["results"]`), but it lands directly in GitHub Code Scanning and most enterprise platforms without translation.
3. **CycloneDX** is an inventory format rather than a vulnerability list. It answers "what's installed" rather than "what's broken." Pairing a Syft CycloneDX SBOM with Grype's JSON or SARIF output is how most pipelines stitch supply-chain visibility together.

A practical pattern for pipelines: keep Grype JSON as the canonical machine-readable record for internal grep/awk/jq workflows, convert to SARIF only at the CI boundary, and retain the CycloneDX SBOM for artifact signing or dependency-graph downstream.

## Verify

To confirm the structures yourself:

1. Replace `GRYPE_TARGET` with a real image or directory, e.g.:
   ```bash
   GRYPE_TARGET=python:3.12-slim jupyter notebook grype-sbom-output-explorer.ipynb
   ```
2. Inspect the three output files in `grype-output-demo/` — open each in a JSON viewer to verify the nesting.
3. Check that the same CVE appears in both `grype.json` and `grype.sarif` by searching for its ID.

If `grype` or `syft` are missing, the notebook still exercises the parsing logic because it writes minimal fixtures before continuing.